In [1]:
import time
import numpy as np
import random
import matplotlib.pyplot as plt
from pathlib import Path
import albumentations as A
import cv2
import time


import torch
from torch import nn
import copy
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from albumentations.pytorch import ToTensorV2

from IPython.display import clear_output
import matplotlib.colors as mcolors
from scipy.ndimage import label
from skimage.segmentation import watershed
from torchvision.utils import make_grid

from Ellipses import *
from plottings import *
from metrics import *
from GRU import *
from gallery_of_failures import *

from load_dataset import *
from rastreadores import *
from treino_gru import *
from utils_mot import *
from sinteticos import *
import os; os.makedirs("checkpoints", exist_ok=True)

%load_ext autoreload
%autoreload 2

In [3]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def desnorm(img, mean=MEAN, std=STD):
    img_plot = img.cpu().permute(1, 2, 0).numpy()
    mean = np.array(mean)
    std = np.array(std)
    img_plot = (img_plot * std) + mean
    return np.clip(img_plot, 0, 1)

# Pipeline para redimensionamento das imagens
train_transform = A.Compose([
    # # Forca image e mask para 256x256
    # A.Resize(height=256, width=256),

    # # Padroniza o espectro de cores como cinza
    # A.ToGray(p=1.0),

    # # Normaliza os pixels para o padrao ImageNet
    # A.Normalize(mean=MEAN, std=STD),

    # Converte os arrays Numpy para Tensores do PyTorch e ajusta as dimensoes ((H, W, C) ==> (C, H, W))
    ToTensorV2()
])

batch_size = 1


# Definindo as sequencias para cada split
# Sao os numeros dos videos do dataset de treino oficial
train_videos = [2, 4, 5, 9, 10]
val_videos = [11, 13]

# Cria o Dataset de Treino (80% dos videos, ordem cronologica)
train_dataset = MOT17Dataset(
    root="features",
    model_base="SDP",
    sequences_list=train_videos,
    transform=train_transform,
    train_dataset=True
)

# Cria o Dataset de Validação (20% dos videos, ordem cronologica)
val_dataset = MOT17Dataset(
    root="features",
    model_base="SDP",
    sequences_list=val_videos,
    transform=train_transform,
    train_dataset=True  # Mantem True para carregar os GTs para calcular as metricas!
)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

In [4]:
current_seq = input("Digite o path:")

frames_by_video = {}
ground_truth_SDP_by_video = {}
dets_SDP_by_video = {}
frames = []
ground_truth_SDP = []
dets_SDP = []
for batch_idx, (images, dets, gts, seq_names, frame_ids) in enumerate(train_loader):
    # O DataLoader insere uma dimensao extra por causa do batch. Extraimos aqui:
    image = images.squeeze(0)        # Formato final: [3, H, W]
    det = dets.squeeze(0)            # Formato final: [Num_Deteccoes, 5]
    gt = gts.squeeze(0)              # Formato final: [Num_Pessoas_Reais, 4]
    
    seq_name = seq_names[0]          # Extrai a string
    frame_id = frame_ids[0].item()   # Extrai o número do frame
    
    # --- DETECTOR DE MUDANÇA DE VIDEO ---
    if seq_name != current_seq:
        # Se tiver terminado um video valido, arquiva os dados dele
        if not current_seq is None:
            frames_by_video[current_seq] = frames
            ground_truth_SDP_by_video[current_seq] = ground_truth_SDP
            dets_SDP_by_video[current_seq] = dets_SDP
            break # Salva so o primeiro video

        frames = []
        ground_truth_SDP = []
        dets_SDP = []
        current_seq = seq_name
    # elif len(frames) >= 50: continue # Salva somente os 50 primeiros frames
    
    frames.append(image.permute(1, 2, 0).numpy())
    gts_str = [
        ', '.join([
            *(row[:-1].int()).numpy().astype(str).tolist(),
            row[-1].numpy().astype(str).tolist()
        ])
        for row in gts[0]
    ]
    ground_truth_SDP.extend(gts_str)
    dets_str = [
        ', '.join([
            *(row[:-1].int()).numpy().astype(str).tolist(),
            row[-1].numpy().astype(str).tolist()
        ])
        for row in dets[0]
    ]
    dets_SDP.extend(dets_str)

In [5]:
def collate_fn(batch):
    """
    Agrupa os dados do batch numa tupla de listas.
    O batch recebe a saída do nosso __getitem__: (image, dets, gts, seq_name, frame_id)
    """
    return tuple(zip(*batch))

In [6]:
import torch
import torchvision
from torch.utils.data import DataLoader

# --- 1. CONFIGURACOES ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
MIN_CONFIDENCE = 0.5  # So aceita predicoes com mais de 50% de certeza

# Instancia o DataLoader (batch_size=1, shuffle=False para processar na ordem)
data_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

# --- 2. CARREGAR MODELO PRE-TREINADO ---
print("Carregando modelo pre-treinado...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LARGURA_VIDEO = 1920
ALTURA_VIDEO = 1080

# 1. Carrega o arquivo campeão do disco
caminho_campeao = "checkpoints/melhor_modelo_janela.pth"#"checkpoints/melhor_modelo_eixo1.pth"
checkpoint = torch.load(caminho_campeao, map_location=device)

# 2. Recria o modelo usando a sua classe do notebook e injeta os pesos
modelo_real = BBoxTrackerRNN(rnn_type=checkpoint['rnn_type'], input_size=4, hidden_size=64).to(device)
modelo_real.load_state_dict(checkpoint['state_dict'])
modelo_real.eval()
modelo_real.to(device)
modelo_real.eval()  # MODO INFERENCIA: Desliga calculo de gradientes e congela camadas (Dropout/BatchNorm)

# Dicionario para guardar as novas deteccoes: dict[seq_name][frame_id] = [bboxes]
novas_deteccoes = {}

print("Iniciando inferencia...")
# --- 3. LOOP DE INFERENCIA ---
with torch.no_grad():
    for batch_idx, (images, dets_publicas, gts, seq_names, frame_ids) in enumerate(data_loader):
        if seq_names[0] != current_seq: continue # Salva somente os 50 primeiros frames de cada video
        # if seq_names[0] != "MOT17-02-SDP": continue # Salva só o primeiro video
        # O DataLoader com collate_fn entrega tuplas. Vamos pegar o primeiro elemento (batch 1)
        image = (images[0] / 255).to(device)
        seq_name = seq_names[0]
        frame_id = frame_ids[0]
        
        # Garante que a chave da sequencia existe no dicionario
        if seq_name not in novas_deteccoes:
            novas_deteccoes[seq_name] = []
        
        # O modelo em eval() recebe uma lista de imagens e retorna uma lista de dicionarios
        predictions = modelo_real([image])[0] 
        
        # Extrai os resultados
        boxes = predictions['boxes'].cpu()   # [N, 4] no formato [x_min, y_min, x_max, y_max]
        labels = predictions['labels'].cpu() # [N] (1 = Pessoa, 3 = Carro, etc no COCO)
        scores = predictions['scores'].cpu() # [N] (de 0.0 a 1.0)
        
        frame_bboxes = []
        
        # --- 4. FILTRAGEM E CONVERSÃO ---
        for i in range(len(boxes)):
            # No dataset COCO (usado pelo torchvision), a classe 1 e 'person'
            if labels[i] == 1 and scores[i] >= MIN_CONFIDENCE:
                
                # Pega as coordenadas originais do Faster R-CNN
                x_min, y_min, x_max, y_max = boxes[i].tolist()
                
                # Converte para o padrão MOT: [x_canto, y_canto, largura, altura]
                w = x_max - x_min
                h = y_max - y_min
                x = x_min
                y = y_min
                conf = scores[i].item()
                
                # Salva no mesmo padrão 9 colunas que definimos antes
                # [frame, id(-1), x, y, w, h, conf, class(1), visibilidade(1.0)]
                bbox_formatada = [frame_id, -1, x, y, w, h, conf, 1.0, 1.0]
                frame_bboxes.append(bbox_formatada)
                
        # Guarda a lista de caixas no frame atual (mesmo que seja uma lista vazia)
        novas_deteccoes[seq_name].append(frame_bboxes)

print("\nInferencia concluida!")

Carregando modelo pre-treinado...


TypeError: BBoxTrackerRNN.__init__() got an unexpected keyword argument 'rnn_type'

In [ ]:
novas_deteccoes